# 🔐 Notebook 3: Pessimistic Locking

Pessimistic locking prevents conflicts by acquiring locks **before** doing work. It's called "pessimistic" because we assume conflicts WILL happen and prevent them upfront.

## Learning Objectives

By the end of this notebook, you'll understand:
- How `SELECT ... FOR UPDATE` works
- Row-level vs table-level locks
- How to prevent deadlocks
- When to use pessimistic locking

## 🔒 How Pessimistic Locking Works

```sql
BEGIN TRANSACTION;

-- Lock the row FIRST (other transactions must wait)
SELECT * FROM concerts WHERE id = 1 FOR UPDATE;

-- Now safely check and update
UPDATE concerts SET available_seats = available_seats - 1 WHERE id = 1;

COMMIT;  -- Release the lock
```

The `FOR UPDATE` clause acquires an **exclusive lock** on the selected rows. Other transactions trying to lock the same rows will **block** until we commit or rollback.

---

🔍 **Open Adminer** at http://localhost:8080 → Click `concerts` → "Select data" to watch locks in action!

In [ ]:
import psycopg2
from concurrent.futures import ThreadPoolExecutor
import time
from threading import Lock

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "contention_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

try:
    conn = get_connection()
    print("✅ Connected to PostgreSQL")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")

In [ ]:
def reset_concert(seats: int = 1):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE concerts SET available_seats = %s WHERE id = 1", (seats,))
    cursor.execute("DELETE FROM tickets WHERE concert_id = 1")
    conn.commit()
    conn.close()

def get_concert_status():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT available_seats FROM concerts WHERE id = 1")
    seats = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM tickets WHERE concert_id = 1")
    tickets = cursor.fetchone()[0]
    conn.close()
    return {"seats": seats, "tickets": tickets}

reset_concert(1)
status = get_concert_status()
print(f"🎫 Concert status: {status['seats']} seats, {status['tickets']} tickets sold")

## 🎫 Fixing the Concert Ticket Problem

Let's implement proper pessimistic locking for our concert ticket scenario.

In [ ]:
event_log = []
log_lock = Lock()

def log_event(user: str, event: str):
    with log_lock:
        timestamp = time.time()
        event_log.append({"time": timestamp, "user": user, "event": event})

def buy_ticket_with_lock(user_id: str) -> dict:
    conn = get_connection()
    cursor = conn.cursor()
    
    try:
        log_event(user_id, "Attempting to acquire lock...")
        
        cursor.execute(
            "SELECT available_seats FROM concerts WHERE id = 1 FOR UPDATE"
        )
        seats = cursor.fetchone()[0]
        
        log_event(user_id, f"Lock acquired! Sees {seats} seats")
        
        time.sleep(0.05)
        
        if seats >= 1:
            cursor.execute(
                "UPDATE concerts SET available_seats = available_seats - 1 WHERE id = 1"
            )
            cursor.execute(
                "INSERT INTO tickets (concert_id, user_id, seat_number, purchase_price) "
                "VALUES (1, %s, 'A1', 150.00)",
                (user_id,)
            )
            conn.commit()
            log_event(user_id, "✅ BOUGHT ticket, releasing lock")
            return {"success": True, "user": user_id}
        else:
            conn.rollback()
            log_event(user_id, "❌ SOLD OUT, releasing lock")
            return {"success": False, "user": user_id, "reason": "sold_out"}
            
    except Exception as e:
        conn.rollback()
        log_event(user_id, f"ERROR: {e}")
        return {"success": False, "user": user_id, "reason": str(e)}
    finally:
        conn.close()

print("✅ Pessimistic locking function created")

In [ ]:
print("🔒 Testing Pessimistic Locking")
print("=" * 60)

reset_concert(1)
event_log.clear()

print("Starting state: 1 seat available")
print("Alice and Bob both try to buy at the same time...\n")

with ThreadPoolExecutor(max_workers=2) as executor:
    f_alice = executor.submit(buy_ticket_with_lock, "Alice")
    f_bob = executor.submit(buy_ticket_with_lock, "Bob")
    
    r_alice = f_alice.result()
    r_bob = f_bob.result()

print("📋 Event Log (ordered by time):")
for event in sorted(event_log, key=lambda x: x["time"]):
    print(f"   [{event['user']:5s}] {event['event']}")

print(f"\n📊 Results:")
print(f"   Alice: {'✅ Got ticket' if r_alice['success'] else '❌ No ticket'}")
print(f"   Bob:   {'✅ Got ticket' if r_bob['success'] else '❌ No ticket'}")

status = get_concert_status()
print(f"\n🎫 Final: {status['seats']} seats, {status['tickets']} tickets sold")

if r_alice['success'] != r_bob['success'] and status['seats'] == 0:
    print("\n✅ SUCCESS! Exactly one person got the ticket!")
    print("   The lock forced the second person to wait.")

## 📊 Stress Test

Let's verify pessimistic locking works under high concurrency.

In [ ]:
def stress_test_locking(num_users: int, available_seats: int):
    reset_concert(available_seats)
    
    def try_buy(user_num):
        return buy_ticket_with_lock(f"User{user_num}")
    
    with ThreadPoolExecutor(max_workers=num_users) as executor:
        futures = [executor.submit(try_buy, i) for i in range(num_users)]
        results = [f.result() for f in futures]
    
    successful = sum(1 for r in results if r["success"])
    status = get_concert_status()
    
    return {
        "users": num_users,
        "seats": available_seats,
        "purchased": successful,
        "final_seats": status["seats"],
        "tickets_in_db": status["tickets"],
        "correct": successful == available_seats and status["seats"] == 0
    }

print("🧪 Stress Testing Pessimistic Locking")
print("=" * 60)
print()

event_log.clear()

test_cases = [
    (5, 3),
    (10, 5),
    (20, 10),
    (50, 20),
]

all_correct = True
for num_users, seats in test_cases:
    result = stress_test_locking(num_users, seats)
    status = "✅" if result["correct"] else "❌"
    if not result["correct"]:
        all_correct = False
    print(f"Users: {result['users']:3d} | Seats: {result['seats']:3d} | "
          f"Sold: {result['purchased']:3d} | Final: {result['final_seats']:3d} | {status}")

print()
if all_correct:
    print("🎉 All tests passed! Pessimistic locking prevents race conditions!")
else:
    print("⚠️ Some tests failed - investigate the issue")

## ⚠️ The Deadlock Problem

Pessimistic locking can cause **deadlocks** if locks are acquired in inconsistent order.

```
Transaction A:                    Transaction B:
1. Lock Alice's account           1. Lock Bob's account
2. Try to lock Bob's account      2. Try to lock Alice's account
   (BLOCKED - B has it!)             (BLOCKED - A has it!)

Both wait forever → DEADLOCK!
```

In [ ]:
def reset_balances():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE accounts SET balance = 1000 WHERE user_id = 'alice'")
    cursor.execute("UPDATE accounts SET balance = 500 WHERE user_id = 'bob'")
    conn.commit()
    conn.close()

def transfer_unsafe(from_user: str, to_user: str, amount: float, delay: float = 0.1):
    conn = get_connection()
    cursor = conn.cursor()
    
    try:
        cursor.execute(
            "SELECT balance FROM accounts WHERE user_id = %s FOR UPDATE",
            (from_user,)
        )
        from_balance = cursor.fetchone()[0]
        
        log_event(f"{from_user}→{to_user}", f"Locked {from_user}, waiting...")
        time.sleep(delay)
        
        cursor.execute(
            "SELECT balance FROM accounts WHERE user_id = %s FOR UPDATE",
            (to_user,)
        )
        to_balance = cursor.fetchone()[0]
        
        log_event(f"{from_user}→{to_user}", f"Locked {to_user}, transferring...")
        
        if from_balance >= amount:
            cursor.execute(
                "UPDATE accounts SET balance = balance - %s WHERE user_id = %s",
                (amount, from_user)
            )
            cursor.execute(
                "UPDATE accounts SET balance = balance + %s WHERE user_id = %s",
                (amount, to_user)
            )
            conn.commit()
            log_event(f"{from_user}→{to_user}", "✅ Transfer complete")
            return True
        else:
            conn.rollback()
            return False
            
    except psycopg2.errors.DeadlockDetected:
        conn.rollback()
        log_event(f"{from_user}→{to_user}", "🔥 DEADLOCK detected!")
        return False
    except Exception as e:
        conn.rollback()
        log_event(f"{from_user}→{to_user}", f"ERROR: {e}")
        return False
    finally:
        conn.close()

print("⚠️ Demonstrating Deadlock Risk")
print("=" * 60)
print()
print("Alice sends $100 to Bob, while Bob sends $50 to Alice...")
print("Both lock their own account first → potential deadlock!\n")

reset_balances()
event_log.clear()

with ThreadPoolExecutor(max_workers=2) as executor:
    f1 = executor.submit(transfer_unsafe, "alice", "bob", 100, 0.1)
    f2 = executor.submit(transfer_unsafe, "bob", "alice", 50, 0.1)
    
    r1 = f1.result()
    r2 = f2.result()

print("📋 Event Log:")
for event in sorted(event_log, key=lambda x: x["time"]):
    print(f"   [{event['user']:12s}] {event['event']}")

print(f"\nResults: Alice→Bob: {r1}, Bob→Alice: {r2}")
print("\n💡 PostgreSQL automatically detects deadlocks and kills one transaction!")

## ✅ Preventing Deadlocks: Ordered Locking

The solution: **always acquire locks in a consistent order** (e.g., by user_id alphabetically).

In [ ]:
def transfer_safe(from_user: str, to_user: str, amount: float, delay: float = 0.1):
    conn = get_connection()
    cursor = conn.cursor()
    
    first_user, second_user = sorted([from_user, to_user])
    
    try:
        cursor.execute(
            "SELECT user_id, balance FROM accounts WHERE user_id = %s FOR UPDATE",
            (first_user,)
        )
        first_balance = cursor.fetchone()[1]
        
        log_event(f"{from_user}→{to_user}", f"Locked {first_user} (ordered)")
        time.sleep(delay)
        
        cursor.execute(
            "SELECT user_id, balance FROM accounts WHERE user_id = %s FOR UPDATE",
            (second_user,)
        )
        second_balance = cursor.fetchone()[1]
        
        log_event(f"{from_user}→{to_user}", f"Locked {second_user} (ordered)")
        
        from_balance = first_balance if first_user == from_user else second_balance
        
        if from_balance >= amount:
            cursor.execute(
                "UPDATE accounts SET balance = balance - %s WHERE user_id = %s",
                (amount, from_user)
            )
            cursor.execute(
                "UPDATE accounts SET balance = balance + %s WHERE user_id = %s",
                (amount, to_user)
            )
            conn.commit()
            log_event(f"{from_user}→{to_user}", "✅ Transfer complete")
            return True
        else:
            conn.rollback()
            return False
            
    except psycopg2.errors.DeadlockDetected:
        conn.rollback()
        log_event(f"{from_user}→{to_user}", "🔥 DEADLOCK (shouldn't happen!)")
        return False
    except Exception as e:
        conn.rollback()
        log_event(f"{from_user}→{to_user}", f"ERROR: {e}")
        return False
    finally:
        conn.close()

print("✅ Demonstrating Deadlock Prevention")
print("=" * 60)
print()
print("Same scenario, but locks acquired in alphabetical order...")
print("Both transactions will lock 'alice' first, then 'bob'\n")

reset_balances()
event_log.clear()

with ThreadPoolExecutor(max_workers=2) as executor:
    f1 = executor.submit(transfer_safe, "alice", "bob", 100, 0.1)
    f2 = executor.submit(transfer_safe, "bob", "alice", 50, 0.1)
    
    r1 = f1.result()
    r2 = f2.result()

print("📋 Event Log:")
for event in sorted(event_log, key=lambda x: x["time"]):
    print(f"   [{event['user']:12s}] {event['event']}")

print(f"\nResults: Alice→Bob: {r1}, Bob→Alice: {r2}")
print("\n✅ No deadlock! Ordered locking ensures consistent acquisition.")

## 🔒 Lock Types in PostgreSQL

PostgreSQL offers several lock modes:

| Lock Mode | Use Case | Blocks |
|-----------|----------|--------|
| `FOR UPDATE` | Update the row | Other FOR UPDATE, FOR SHARE |
| `FOR NO KEY UPDATE` | Update non-key columns | Only FOR UPDATE |
| `FOR SHARE` | Read but prevent updates | FOR UPDATE, FOR NO KEY UPDATE |
| `FOR KEY SHARE` | Prevent key changes | Only FOR UPDATE |

`FOR UPDATE NOWAIT` - Fail immediately if lock unavailable
`FOR UPDATE SKIP LOCKED` - Skip rows that are locked

In [ ]:
def buy_ticket_nowait(user_id: str) -> dict:
    conn = get_connection()
    cursor = conn.cursor()
    
    try:
        cursor.execute(
            "SELECT available_seats FROM concerts WHERE id = 1 FOR UPDATE NOWAIT"
        )
        seats = cursor.fetchone()[0]
        
        time.sleep(0.1)
        
        if seats >= 1:
            cursor.execute(
                "UPDATE concerts SET available_seats = available_seats - 1 WHERE id = 1"
            )
            conn.commit()
            return {"success": True, "user": user_id}
        else:
            conn.rollback()
            return {"success": False, "user": user_id, "reason": "sold_out"}
            
    except psycopg2.errors.LockNotAvailable:
        conn.rollback()
        return {"success": False, "user": user_id, "reason": "lock_unavailable"}
    except Exception as e:
        conn.rollback()
        return {"success": False, "user": user_id, "reason": str(e)}
    finally:
        conn.close()

print("🚀 Testing FOR UPDATE NOWAIT")
print("=" * 50)
print()
print("NOWAIT fails immediately instead of waiting for lock...\n")

reset_concert(1)

with ThreadPoolExecutor(max_workers=2) as executor:
    f1 = executor.submit(buy_ticket_nowait, "Alice")
    f2 = executor.submit(buy_ticket_nowait, "Bob")
    
    r1 = f1.result()
    r2 = f2.result()

print(f"Alice: {r1}")
print(f"Bob:   {r2}")
print()
print("💡 NOWAIT is useful when you'd rather fail fast than wait!")

## 🧪 Quick Quiz

1. **What's the difference between FOR UPDATE and FOR SHARE?**

2. **How do you prevent deadlocks with pessimistic locking?**

3. **When would you use FOR UPDATE NOWAIT?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. FOR UPDATE vs FOR SHARE:")
print("   - FOR UPDATE: Exclusive lock, blocks other writers AND readers")
print("   - FOR SHARE: Shared lock, blocks writers but allows other readers")
print("   - Use FOR SHARE when you need consistent read without updating")
print()
print("2. Preventing deadlocks:")
print("   - Always acquire locks in a CONSISTENT ORDER")
print("   - Sort resources by some key (ID, name) before locking")
print("   - This prevents circular wait conditions")
print()
print("3. When to use NOWAIT:")
print("   - When failing fast is better than waiting")
print("   - High-contention scenarios where waiting is wasteful")
print("   - API endpoints with strict timeout requirements")

## 📚 Summary

### What We Learned

1. **SELECT ... FOR UPDATE** acquires exclusive locks on rows
2. **Pessimistic locking** assumes conflicts will happen
3. **Deadlocks** occur when locks are acquired in inconsistent order
4. **Ordered locking** prevents deadlocks by consistent acquisition order
5. **NOWAIT/SKIP LOCKED** provide alternatives to blocking

### When to Use Pessimistic Locking

| Scenario | Use Pessimistic? |
|----------|------------------|
| High contention (many users, few resources) | ✅ Yes |
| Low contention (rare conflicts) | ❌ No (use optimistic) |
| Short transactions | ✅ Yes |
| Long transactions | ❌ No (blocks too long) |
| Predictable access patterns | ✅ Yes |

### Next Up: Optimistic Concurrency Control

In the next notebook, we'll learn about **optimistic concurrency** - assuming conflicts are rare and detecting them after they occur!